In [ ]:

# Uncomment if needed:
# %pip install -q transformers scikit-learn

In [ ]:
import json
import random
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# Fix random seeds for reproducible clustering/sampling
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
#data induction
DATA_DIR = (
    Path(__file__)
    / "swedish training"
)

DATASET = "dev"

DATA_PATH = DATA_DIR / {
    "dev": "dev.zip",
    "sample": "input_data_development (1).zip",
}[DATASET]

with ZipFile(DATA_PATH) as archive:
    files = [
        name for name in archive.namelist()
        if Path(name).name == "usages.jsonl"
        and not name.startswith("__MACOSX/")
    ]
    if len(files) != 1:
        raise ValueError(f"Expected one usages.jsonl; found {files}")

    with archive.open(files[0]) as stream:
        usages_df = pd.read_json(stream, lines=True, dtype=False)

usages_df["language"] = "SV"
usages_df[["start", "end"]] = usages_df[["start", "end"]].astype(int)

if usages_df.duplicated(["word", "sentence_id"]).any():
    raise ValueError("Duplicate usage IDs found.")

print(f"{DATASET}: {len(usages_df):,} usages | "
      f"{usages_df['word'].nunique()} words")

display(usages_df.head())

dev: 1,983 usages | 10 words


,word,sentence_id,text,pos,period_label,start,end,year,language
0,kvinna,137d9286_62_69,nesliga lif medelst sjelfmord ; och ett stort ...,NN,1880-1883,62,69,1883,SV
1,kvinna,b382a5e9_92_98,"— Jag sov ; jag väcktes av en kylande känsla ,...",NN,1912-1915,92,98,1914,SV
2,kvinna,28ffffe7_42_50,"orden , så mycken fasthet öfver den gamla kvin...",NN,1880-1883,42,50,1883,SV
3,kvinna,3754909a_117_124,Dessa klubbar ha anordnat sällskapliga samkväm...,NN,1912-1915,117,124,1913,SV
4,kvinna,59ebb00_32_39,Den avvisningshotade eritranska kvinnan Abiba ...,NN,2006-2007,32,39,2007,SV


In [ ]:
#checking target spans
df = usages_df.copy().reset_index(drop=True)
df["row_id"] = np.arange(len(df))

df["target_form"] = [
    row.text[row.start:row.end]
    for row in df.itertuples()
]

valid_spans = (
    (df["start"] >= 0)
    & (df["end"] > df["start"])
    & (df["end"] <= df["text"].str.len())
    & df["target_form"].str.strip().ne("")
)

if not valid_spans.all():
    display(df.loc[~valid_spans, [
        "word", "sentence_id", "start", "end", "target_form", "text"
    ]])
    raise ValueError("Invalid target spans—inspect the rows above.")

print(f"Ready: {len(df):,} usages | {df['word'].nunique()} words")

display(df[[
    "word", "sentence_id", "period_label", "target_form", "text"
]].head())

Ready: 1,983 usages | 10 words


,word,sentence_id,period_label,target_form,text
0,kvinna,137d9286_62_69,1880-1883,kvinnor,nesliga lif medelst sjelfmord ; och ett stort ...
1,kvinna,b382a5e9_92_98,1912-1915,kvinna,"— Jag sov ; jag väcktes av en kylande känsla ,..."
2,kvinna,28ffffe7_42_50,1880-1883,kvinnans,"orden , så mycken fasthet öfver den gamla kvin..."
3,kvinna,3754909a_117_124,1912-1915,kvinnor,Dessa klubbar ha anordnat sällskapliga samkväm...
4,kvinna,59ebb00_32_39,2006-2007,kvinnan,Den avvisningshotade eritranska kvinnan Abiba ...


Subtask 1a

In [ ]:
#Bert based vectors
BERT_MODEL_NAME = "KB/bert-base-swedish-cased"
device = "cuda" if torch.cuda.is_available() else "cpu"

bert_tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_NAME, use_fast=True
)

bert_model = AutoModelForMaskedLM.from_pretrained(
    BERT_MODEL_NAME
).to(device)

bert_model.eval()

print(f"Swedish BERT loaded on {device}")

#XL LEZEME
# import gc
# import torch
# from transformers import AutoTokenizer, AutoModel

# # # Free GPU space for XL-LEXEME.
# # if "bert_model" in globals():
# #     bert_model.to("cpu")

# # gc.collect()
# # if torch.cuda.is_available():
# #     torch.cuda.empty_cache()

XL_MODEL_NAME = "pierluigic/xl-lexeme"
xl_device = "cuda" if torch.cuda.is_available() else "cpu"

xl_tokenizer = AutoTokenizer.from_pretrained(
    XL_MODEL_NAME,
    use_fast=True,
)

# xl_model = AutoModel.from_pretrained(
#     XL_MODEL_NAME,
# ).to(xl_device)

# xl_model.eval()

# TESTING SMALLER XL-LEXEME
from sentence_transformers import SentenceTransformer
xl_model = SentenceTransformer("pierluigic/xl-lexeme", device=xl_device)
# TESTING SMALLER XL-LEXEME

print("Device:", xl_device)
print("Embedding width:", xl_model.config.hidden_size)

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/399k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

... (19 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Device: cuda
Embedding width: 1024


In [ ]:
#only if not using BERT with masked word nor XL LEXEME
BATCH_SIZE = 16
MAX_CONTENT = bert_model.config.max_position_embeddings - 2
bert_tokenizer.padding_side = "right"


def prepare_target(row):
    """Tokenize text, locate subword tokens overlapping [start, end], and
    center a max-length window around the target span if the sentence is too long
    """
    tokens = bert_tokenizer(
        row.text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        verbose=False,
    )

    # Find all token indices whose character span overlaps the target span
    positions = [
        i for i, (start, end) in enumerate(tokens["offset_mapping"])
        if start < row.end and end > row.start
    ]

    if not positions:
        raise ValueError(f"No target tokens: {row.sentence_id}")

    span_length = positions[-1] - positions[0] + 1
    if span_length > MAX_CONTENT:
        raise ValueError(f"Target exceeds model window: {row.sentence_id}")

    # Keep the target inside the window for long sentences
    left = max(0, min(
        positions[0] - (MAX_CONTENT - span_length) // 2,
        len(tokens["input_ids"]) - MAX_CONTENT,
    ))

    ids = (
        [bert_tokenizer.cls_token_id]
        + tokens["input_ids"][left:left + MAX_CONTENT]
        + [bert_tokenizer.sep_token_id]
    )

    # Shift target token positions to account for window offset and leading [CLS]
    return {"input_ids": ids}, [p - left + 1 for p in positions]


vectors = []

for i in tqdm(range(0, len(df), BATCH_SIZE), desc="BERT: original target"):
    batch = df.iloc[i:i + BATCH_SIZE]
    prepared = [prepare_target(row) for row in batch.itertuples()]

    inputs = bert_tokenizer.pad(
        [item[0] for item in prepared],
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        # Extract last hidden state from base encoder (without MLM head)
        hidden = bert_model.base_model(**inputs).last_hidden_state

        # Mean-pool across the subword tokens corresponding to the target word
        for j, (_, positions) in enumerate(prepared):
            vector = hidden[j, positions].mean(dim=0)
            vectors.append(vector.cpu().numpy())

df["bert_target"] = vectors

print("Embedding shape:", np.stack(vectors).shape)

XL_BATCH_SIZE = 8
XL_MAX_LENGTH = 128
xl_tokenizer.padding_side = "right"

def prepare_xl_target(row):
    text = row.text
    start, end = int(row.start), int(row.end)

    if not 0 <= start < end <= len(text):
        raise ValueError(f"Invalid target span: {row.sentence_id}")

    # Tokenize segments separately, following the author's approach.
    left = xl_tokenizer.tokenize(text[:start])
    marked_target = (
        xl_tokenizer.tokenize("<t>")
        + xl_tokenizer.tokenize(text[start:end])
        + xl_tokenizer.tokenize("</t>")
    )
    right = xl_tokenizer.tokenize(text[end:])

    content_budget = XL_MAX_LENGTH - 2  # Reserve start/end tokens.
    if len(marked_target) > content_budget:
        raise ValueError(f"Target too long: {row.sentence_id}")

    # Truncate outer context equally on both sides if sequence exceeds budget
    if len(left) + len(marked_target) + len(right) > content_budget:
        context_each_side = (content_budget - len(marked_target)) // 2
        left = left[-context_each_side:] if context_each_side else []
        right = right[:context_each_side]

    tokens = (
        [xl_tokenizer.cls_token]
        + left + marked_target + right
        + [xl_tokenizer.sep_token]
    )

    return {
        "input_ids": xl_tokenizer.convert_tokens_to_ids(tokens),
        "attention_mask": [1] * len(tokens),
    }

xl_batches = []

for start in tqdm(
    range(0, len(df), XL_BATCH_SIZE),
    desc="XL-LEXEME: target",
):
    batch = df.iloc[start:start + XL_BATCH_SIZE]
    prepared = [prepare_xl_target(row) for row in batch.itertuples()]

    inputs = xl_tokenizer.pad(
        prepared,
        padding=True,
        return_tensors="pt",
    ).to(xl_device)

    with torch.inference_mode():
        # hidden = xl_model(**inputs).last_hidden_state

        # TESTING SMALLER XL-LEXEME
        hidden = xl_model[0].auto_model(**inputs).last_hidden_state
        # TESTING SMALLER XL-LEXEME

        # Mean over all non-padding tokens in the marked context.
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)

    xl_batches.append(pooled.cpu().numpy())

xl_embeddings = np.concatenate(xl_batches, axis=0)

assert xl_embeddings.shape == (len(df), xl_model.config.hidden_size)
assert np.isfinite(xl_embeddings).all()

df["xl_target"] = list(xl_embeddings)
print("Embedding shape:", xl_embeddings.shape)

BERT: original target:   0%|          | 0/124 [00:00<?, ?it/s]

Embedding shape: (1983, 768)


XL-LEXEME: target:   0%|          | 0/248 [00:00<?, ?it/s]

Embedding shape: (1983, 1024)


In [ ]:
#masked generation (comment out if using XL LEXEME or target BERT)
TOP_K = 5

masked_df = df[["sentence_id", "text", "end"]].copy()
masked_df["text"] = [
    r.text[:r.end] + f" och {bert_tokenizer.mask_token}" + r.text[r.end:]
    for r in df.itertuples()
]
masked_df["start"] = df["end"] + len(" och ")
masked_df["end"] = masked_df["start"] + len(bert_tokenizer.mask_token)

# Allow only standalone alphabetic tokens, excluding WordPiece fragments.
allowed_ids = torch.tensor([
    token_id
    for token, token_id in bert_tokenizer.get_vocab().items()
    if token.isalpha()
], device=device)

x_vectors = []
candidate_words = []
candidate_probs = []

for i in tqdm(range(0, len(masked_df), BATCH_SIZE), desc="BERT: predicted X"):
    batch = masked_df.iloc[i:i + BATCH_SIZE]
    prepared = [prepare_target(row) for row in batch.itertuples()]

    if any(len(positions) != 1 for _, positions in prepared):
        raise ValueError("Expected exactly one token at each inserted mask.")

    inputs = bert_tokenizer.pad(
        [item[0] for item in prepared],
        padding=True,
        return_tensors="pt",
    ).to(device)

    mask_positions = torch.tensor(
        [positions[0] for _, positions in prepared], device=device
    )
    batch_rows = torch.arange(len(prepared), device=device)

    if not torch.all(
        inputs["input_ids"][batch_rows, mask_positions]
        == bert_tokenizer.mask_token_id
    ):
        raise ValueError("The selected positions do not point to masks.")

    with torch.inference_mode():
        # Predict probability distribution over the vocabulary at the mask
        logits = bert_model(**inputs).logits[batch_rows, mask_positions]
        probabilities = logits.softmax(dim=-1)

        # Select top-K allowed alphabetic candidates
        top_probs, top_indices = probabilities[:, allowed_ids].topk(TOP_K)
        top_ids = allowed_ids[top_indices]

        # Replace the mask with the best candidate, then embed that token.
        inputs["input_ids"][batch_rows, mask_positions] = top_ids[:, 0]
        hidden = bert_model.base_model(**inputs).last_hidden_state
        x_vectors.extend(hidden[batch_rows, mask_positions].cpu().numpy())

    candidate_words.extend([
        bert_tokenizer.convert_ids_to_tokens(ids)
        for ids in top_ids.cpu().tolist()
    ])
    candidate_probs.extend(top_probs.cpu().tolist())

df["bert_predicted_x"] = x_vectors
df["X_candidates"] = candidate_words
df["X_candidate_probs"] = candidate_probs
df["predicted_X"] = [words[0] for words in candidate_words]

# Save the completed context and X's character span for XL-LEXEME later.
df["text_with_X"] = [
    r.text[:r.end] + f" och {r.predicted_X}" + r.text[r.end:]
    for r in df.itertuples()
]
df["x_start"] = df["end"] + len(" och ")
df["x_end"] = df["x_start"] + df["predicted_X"].str.len()

print("Predicted-X shape:", np.stack(x_vectors).shape)
display(df[["word", "target_form", "predicted_X", "X_candidates"]].head(10))

BERT: predicted X:   0%|          | 0/124 [00:00<?, ?it/s]

Predicted-X shape: (1983, 768)


,word,target_form,predicted_X,X_candidates
0,kvinna,kvinnor,män,"[män, barn, flickor, döttrar, bröder]"
1,kvinna,kvinna,hon,"[hon, som, jag, den, lion]"
2,kvinna,kvinnans,mannens,"[mannens, männens, hennes, barnets, flickans]"
3,kvinna,kvinnor,män,"[män, barn, flickor, herrar, döttrar]"
4,kvinna,kvinnan,flickan,"[flickan, dottern, barnet, mannen, fotografen]"
5,kvinna,kvinnornas,barnens,"[barnens, ungdomens, männens, arbetarnas, lära..."
6,kvinna,Kvinnan,mannen,"[mannen, kvinnan, flickan, föraren, barnet]"
7,kvinna,kvinnan,läkaren,"[läkaren, hon, mannen, kvinnan, han]"
8,kvinna,kvinna,man,"[man, studerande, dotter, flicka, kamrat]"
9,kvinna,kvinnor,män,"[män, barn, flickor, tjejer, äldre]"


In [ ]:
# Reuse BERT's predicted words and mark X as the XL target
xl_x_df = df.assign(
    text=df["text_with_X"],
    start=df["x_start"],
    end=df["x_end"],
)

xl_x_batches = []
xl_model.eval()

for start in tqdm(
    range(0, len(xl_x_df), XL_BATCH_SIZE),
    desc="XL-LEXEME: predicted X",
):
    batch = xl_x_df.iloc[start:start + XL_BATCH_SIZE]
    prepared = [prepare_xl_target(row) for row in batch.itertuples()]

    inputs = xl_tokenizer.pad(
        prepared, padding=True, return_tensors="pt"
    ).to(xl_device)

    with torch.inference_mode():
        # hidden = xl_model(**inputs).last_hidden_state

        # TESTING SMALLER XL-LEXEME
        hidden = xl_model[0].auto_model(**inputs).last_hidden_state
        # TESTING SMALLER XL-LEXEME
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (
            (hidden * mask).sum(dim=1)
            / mask.sum(dim=1).clamp_min(1)
        )

    xl_x_batches.append(pooled.cpu().numpy())

xl_x_embeddings = np.concatenate(xl_x_batches, axis=0)

assert len(xl_x_embeddings) == len(df)
assert np.isfinite(xl_x_embeddings).all()

df["xl_predicted_x"] = list(xl_x_embeddings)
print("Embedding shape:", xl_x_embeddings.shape)

XL-LEXEME: predicted X:   0%|          | 0/248 [00:00<?, ?it/s]

Embedding shape: (1983, 1024)


In [ ]:
df #contains both vector of original word and masked word

,word,sentence_id,text,pos,period_label,start,end,year,language,row_id,...,bert_target,xl_target,bert_predicted_x,X_candidates,X_candidate_probs,predicted_X,text_with_X,x_start,x_end,xl_predicted_x
0,kvinna,137d9286_62_69,nesliga lif medelst sjelfmord ; och ett stort ...,NN,1880-1883,62,69,1883,SV,0,...,"[-0.3078088, -0.22193699, 0.3180581, 0.352196,...","[-1.454266, 0.11998356, -0.65147305, 0.1453139...","[-1.3956956, -0.25454265, -0.7530624, -0.52766...","[män, barn, flickor, döttrar, bröder]","[0.5277261734008789, 0.32105398178100586, 0.10...",män,nesliga lif medelst sjelfmord ; och ett stort ...,74,77,"[-1.3691194, 0.29074043, -1.3648174, -0.591182..."
1,kvinna,b382a5e9_92_98,"— Jag sov ; jag väcktes av en kylande känsla ,...",NN,1912-1915,92,98,1914,SV,1,...,"[0.12105333, 0.15957108, -0.23771894, -0.10086...","[-1.2984182, 0.14616948, -1.0349121, -0.565765...","[0.00482953, 0.8147716, -0.31288764, -0.026239...","[hon, som, jag, den, lion]","[0.5359036326408386, 0.24406541883945465, 0.01...",hon,"— Jag sov ; jag väcktes av en kylande känsla ,...",103,106,"[-0.9245052, -0.37213928, -1.0352257, 0.240941..."
2,kvinna,28ffffe7_42_50,"orden , så mycken fasthet öfver den gamla kvin...",NN,1880-1883,42,50,1883,SV,2,...,"[0.29385206, 0.4028187, 0.2195406, 0.014011037...","[-0.90987283, -0.044490077, -1.217343, -0.3257...","[-0.68670356, 0.19119635, -0.79294324, -0.3826...","[mannens, männens, hennes, barnets, flickans]","[0.1829071044921875, 0.0671597346663475, 0.059...",mannens,"orden , så mycken fasthet öfver den gamla kvin...",55,62,"[-0.5180709, -0.5893559, -1.1726696, -1.540059..."
3,kvinna,3754909a_117_124,Dessa klubbar ha anordnat sällskapliga samkväm...,NN,1912-1915,117,124,1913,SV,3,...,"[0.0779299, 0.30330408, 0.3639017, 0.116927, -...","[-1.4486976, -0.04322303, -0.47132903, -0.2649...","[-0.6937958, 0.3709296, -0.39423105, -0.463035...","[män, barn, flickor, herrar, döttrar]","[0.4055851101875305, 0.3066297173500061, 0.189...",män,Dessa klubbar ha anordnat sällskapliga samkväm...,129,132,"[-1.1470668, 0.17080435, -1.3343664, -0.785311..."
4,kvinna,59ebb00_32_39,Den avvisningshotade eritranska kvinnan Abiba ...,NN,2006-2007,32,39,2007,SV,4,...,"[-0.2247178, -0.8759687, -0.3949067, 0.2464650...","[-1.3894095, 0.23398249, -1.1168133, -0.076379...","[-0.4672209, -0.9104784, -1.0631952, 0.8151578...","[flickan, dottern, barnet, mannen, fotografen]","[0.2377144992351532, 0.07840375602245331, 0.07...",flickan,Den avvisningshotade eritranska kvinnan och fl...,44,51,"[-2.3654544, 0.64493805, -1.4228811, 0.3294792..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1978,motiv,7ff26f2a_79_84,"Er fördom emot honom är så mycket starkare , s...",NN,1880-1883,79,84,1880,SV,1978,...,"[0.12556878, -0.44409233, -0.12388238, 0.10373...","[0.48639688, -1.2429774, -0.24640867, -0.02299...","[0.2419794, -0.41608602, -0.2935971, -0.572031...","[egenskaper, idéer, tankar, begrepp, åsikter]","[0.1701333373785019, 0.09054328501224518, 0.08...",egenskaper,"Er fördom emot honom är så mycket starkare , s...",89,99,"[0.7534149, -0.5086739, -0.42667943, 0.3497703..."
1979,motiv,91aa07c7_150_155,Männen fälldes för mordförsök – en man och en ...,NN,2010-2011,150,155,2010,SV,1979,...,"[-0.4556172, 0.56736165, -0.4232942, 0.2046009...","[0.02070389, -2.108411, 0.26003706, -0.1943701...","[-0.7248217, 0.4049076, -0.89847016, 0.0928145...","[motiv, omfattning, våld, karaktär, svartsjuka]","[0.15042951703071594, 0.11334027349948883, 0.0...",motiv,Männen fälldes för mordförsök – en man och en ...,160,165,"[-0.22078596, -1.9568018, 0.087523766, -0.7558..."
1980,motiv,bcf19e0a_200_205,Att Napoleon fråntog Bernadotte en af hans div...,NN,1896-1899,200,205,1899,SV,1980,...,"[0.40487176, -0.016079277, 0.28664982, 0.25260...","[0.06061815, -1.8955983, 0.30507153, -0.378915...","[0.32590947, 0.46750546, -0.17029527, 0.646930...","[ideal, idéer, dylikt, personer, motiv]","[0.02984035573899746, 0.01823113113641739, 0.0...",ide

In [ ]:
%pip install -q bcubed

import bcubed

if DATASET != "dev":
    raise ValueError("Load dev and run its embedding cells first.")

with ZipFile(DATA_DIR / "dev.zip") as archive:
    with archive.open("SV/subtask1.jsonl") as stream:
        gold_df = pd.read_json(stream, lines=True, dtype=False)

keys = ["word", "sentence_id", "period_label"]

# Match labels to the exact order of our embeddings.
aligned_gold = df[keys].merge(
    gold_df[keys + ["label"]],
    on=keys,
    how="left",
    sort=False,
    validate="one_to_one",
    indicator=True,
)

if (
    len(df) != len(gold_df)
    or not aligned_gold["_merge"].eq("both").all()
):
    raise ValueError("Current embeddings do not match the full dev set.")

gold_labels = aligned_gold["label"].tolist()

def overlapping_b3(predicted, gold):
    predicted = {i: set(labels) for i, labels in enumerate(predicted)}
    gold = {i: set(labels) for i, labels in enumerate(gold)}

    precision = bcubed.precision(predicted, gold)
    recall = bcubed.recall(predicted, gold)
    return bcubed.fscore(precision, recall)

print(f"Ready to evaluate {len(gold_labels)} dev usages.")

Ready to evaluate 1983 dev usages.


In [ ]:
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

K_VALUES = range(1, 51)
DP_MAX_COMPONENTS = 20
DP_PRIORS = [0.1, 1.0, 10.0]
THRESHOLDS = [0.10, 0.20, 0.25, 0.30, 0.40]

branches = {"xl_predicted_x": "xl_predicted_x", "xl_target": "xl_target", "target": "bert_target", "predicted_x": "bert_predicted_x"}

settings = (
    [("GMM", k) for k in K_VALUES]
    + [("DPMM", prior) for prior in DP_PRIORS]
)
groups = df.groupby("word", sort=False).indices

def assign_senses(probabilities, threshold):
    labels = []
    for scores in probabilities:
        selected = np.flatnonzero(scores >= threshold)
        if selected.size == 0:
            selected = np.array([scores.argmax()])
        labels.append((selected + 1).tolist())
    return labels

records = []
candidate_predictions = {}

progress = tqdm(
    total=len(branches) * len(groups) * len(settings),
    desc="Comparing clustering settings",
)

for branch, embedding_col in branches.items():
    for word, positions in groups.items():
        group = df.iloc[positions]

        # Embeddings only: gold labels are used below for scoring.
        X = normalize(np.stack(group[embedding_col].tolist()))
        dimensions = min(20, len(group) - 1, X.shape[1])
        X = PCA(
            n_components=dimensions,
            svd_solver="full",
        ).fit_transform(X)

        word_gold = [gold_labels[p] for p in positions]

        for method, value in settings:
            common = dict(
                covariance_type="diag",
                reg_covar=1e-5,
                n_init=3,
                max_iter=1000,
                random_state=SEED,
            )

            if method == "GMM":
                setting = f"K={value}"
                model = GaussianMixture(
                    n_components=value, **common
                )
            else:
                setting = f"prior={value:g}"
                model = BayesianGaussianMixture(
                    n_components=min(DP_MAX_COMPONENTS, len(group)),
                    weight_concentration_prior_type="dirichlet_process",
                    weight_concentration_prior=value,
                    **common,
                )

            model.fit(X)
            if not model.converged_:
                raise RuntimeError(
                    f"Did not converge: {word}, {branch}, "
                    f"{method}, {setting}"
                )

            probabilities = model.predict_proba(X)

            for threshold in THRESHOLDS:
                labels = assign_senses(probabilities, threshold)
                config = (branch, method, setting, threshold)

                if config not in candidate_predictions:
                    candidate_predictions[config] = [None] * len(df)

                for position, senses in zip(positions, labels):
                    candidate_predictions[config][position] = senses

                records.append({
                    "branch": branch,
                    "method": method,
                    "setting": setting,
                    "threshold": threshold,
                    "word": word,
                    "b3": overlapping_b3(labels, word_gold),
                    "clusters_used": len({
                        sense for senses in labels for sense in senses
                    }),
                    "multi_pct": 100 * np.mean([
                        len(senses) > 1 for senses in labels
                    ]),
                })

            progress.update(1)

progress.close()

# Each word receives equal weight.
results = pd.DataFrame(records)
config_cols = ["branch", "method", "setting", "threshold"]

ranking = (
    results.groupby(config_cols, as_index=False)
    .agg(
        dev_b3=("b3", "mean"),
        mean_clusters=("clusters_used", "mean"),
        mean_multi_pct=("multi_pct", "mean"),
    )
    .sort_values("dev_b3", ascending=False)
)

best = ranking.drop_duplicates(["branch", "method"]).reset_index(drop=True)

# Keep predictions for the four winning configurations.
predictions = df[["word", "sentence_id", "period_label"]].copy()

for row in best.itertuples(index=False):
    config = (row.branch, row.method, row.setting, row.threshold)
    predictions[f"{row.branch}_{row.method}"] = candidate_predictions[config]

display(best.round(4))

Comparing clustering settings:   0%|          | 0/2120 [00:00<?, ?it/s]

,branch,method,setting,threshold,dev_b3,mean_clusters,mean_multi_pct
0,xl_target,GMM,K=1,0.30,0.7563,1.0,0.0000
1,target,GMM,K=1,0.10,0.7563,1.0,0.0000
2,xl_predicted_x,GMM,K=1,0.25,0.7563,1.0,0.0000
3,predicted_x,GMM,K=1,0.20,0.7563,1.0,0.0000
4,xl_target,DPMM,prior=1,0.10,0.7148,10.5,1.9680
5,target,DPMM,prior=1,0.10,0.5566,12.9,0.1493
6,predicted_x,DPMM,prior=1,0.10,0.4064,15.0,0.2681
7,xl_predicted_x,DPMM,prior=0.1,0.10,0.3560,15.5,0.7793


In [ ]:
import tempfile
from pathlib import Path

all_score_rows = []
config_cols = ["branch", "method", "setting", "threshold"]

with tempfile.TemporaryDirectory() as tmp:
    gold_path = Path(tmp) / "gold.jsonl"
    pred_path = Path(tmp) / "pred.jsonl"

    gold_df.to_json(
        gold_path, orient="records", lines=True, force_ascii=False
    )

    submission = df[["word", "period_label", "sentence_id"]].copy()

    for config, labels in tqdm(
        candidate_predictions.items(),
        total=len(candidate_predictions),
        desc="Evaluating all configurations",
    ):
        submission["label"] = labels
        submission.to_json(
            pred_path, orient="records", lines=True, force_ascii=False
        )

        score_1a, score_1b = evaluate_subtask1(gold_path, pred_path)

        all_score_rows.append({
            **dict(zip(config_cols, config)),
            "B3_F1": score_1a["fcubed"],
            "subtask1b_rho": score_1b,
        })

# All configurations, ordered by Subtask1b score
all_subtask_scores = (
    pd.DataFrame(all_score_rows)
    .sort_values(["subtask1b_rho", "B3_F1"], ascending=False)
    .reset_index(drop=True)
)

# Best Subtask1b configuration per branch and method
best_1b = (
    all_subtask_scores
    .drop_duplicates(["branch", "method"])
    .reset_index(drop=True)
)

display(best_1b.round(4))

Evaluating all configurations:   0%|          | 0/1060 [00:00<?, ?it/s]

NameError: name 'evaluate_subtask1' is not defined

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize

def cluster_with_word_threshold(
    embeddings, quantile=0.50, overlap_margin=0.05
):
    X = normalize(np.asarray(embeddings, dtype=np.float64))

    if len(X) == 1:
        return [[1]], 0.0, 1

    distances = np.clip(1.0 - X @ X.T, 0.0, 2.0)
    np.fill_diagonal(distances, 0.0)

    # Compute a word-specific cutoff from distinct usage pairs.
    pair_distances = distances[np.triu_indices(len(X), k=1)]
    cutoff = float(np.quantile(pair_distances, quantile))

    base_labels = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=float(np.nextafter(cutoff, np.inf)),
        compute_full_tree=True,
    ).fit_predict(distances)

    cluster_ids = np.unique(base_labels)

    # Average distance from each usage to each base cluster.
    cluster_distances = np.column_stack([
        distances[:, base_labels == cluster_id].mean(axis=1)
        for cluster_id in cluster_ids
    ])

    assignments = []

    for i, usage_distances in enumerate(cluster_distances):
        extra_clusters = cluster_ids[
            usage_distances <= cutoff + overlap_margin
        ]

        # Retain the base cluster and allow additional memberships.
        selected = np.union1d(extra_clusters, [base_labels[i]])
        assignments.append((selected + 1).astype(int).tolist())

    return assignments, cutoff, len(cluster_ids)


WORD_QUANTILE = 0.50
OVERLAP_MARGIN = 0.05

threshold_predictions = [None] * len(df)
threshold_word_rows = []

# Cluster all periods together, separately for each word.
for word, positions in df.groupby("word", sort=False).indices.items():
    embeddings = np.stack(
        df.iloc[positions]["xl_target"].tolist()
    )

    labels, cutoff, cluster_count = cluster_with_word_threshold(
        embeddings,
        quantile=WORD_QUANTILE,
        overlap_margin=OVERLAP_MARGIN,
    )

    for position, senses in zip(positions, labels):
        threshold_predictions[position] = senses

    threshold_word_rows.append({
        "word": word,
        "distance_cutoff": cutoff,
        "base_clusters": cluster_count,
        "multi_pct": 100 * np.mean([
            len(senses) > 1 for senses in labels
        ]),
    })

threshold_submission = df[
    ["word", "period_label", "sentence_id"]
].copy()
threshold_submission["label"] = threshold_predictions

threshold_word_stats = pd.DataFrame(threshold_word_rows)
display(threshold_word_stats.round(4))

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    gold_path = Path(tmp) / "gold.jsonl"
    pred_path = Path(tmp) / "pred.jsonl"

    gold_df.to_json(
        gold_path, orient="records", lines=True, force_ascii=False
    )
    threshold_submission.to_json(
        pred_path, orient="records", lines=True, force_ascii=False
    )

    score_1a, score_1b = evaluate_subtask1(gold_path, pred_path)

threshold_initial_scores = {
    "B3_F1": score_1a["fcubed"],
    "subtask1b_rho": score_1b,
}

print(threshold_initial_scores)